# Distributed Data Processing with Ray Data

This notebook demonstrates how to process the January 2019 NYC Yellow Taxi dataset with Ray Data and compare the same end-to-end aggregation with pandas. The dataset contains approximately 7.7 million trips, making it large enough to illustrate Ray's block-based parallel execution while still fitting in memory on many development machines. We will download the data, configure a local Ray runtime, calculate the mean trip total by passenger count, and discuss when distributed processing is worth its additional overhead.

## Learning objectives

By the end of this notebook, you will be able to:

- Load a large CSV as a partitioned Ray Dataset.
- Control local CPU resources and Ray Data logging.
- Resolve an ambiguous CSV column type with PyArrow.
- Execute a distributed group-by aggregation.
- Compare Ray and pandas using equivalent end-to-end workloads.

In [ ]:
# !pip install -U "ray[data]"

## Download the dataset

The release asset is gzip-compressed to reduce download time. Run the following commands from the notebook's working directory:

```bash
mkdir -p data

curl -L --fail --retry 3 \
    -o data/yellow_tripdata_2019-01.csv.gz \
    https://github.com/DataTalksClub/nyc-tlc-data/releases/download/yellow/yellow_tripdata_2019-01.csv.gz
```

Check the compressed size, decompress the file while retaining the original archive, and then inspect the resulting CSV:

```bash
du -h data/yellow_tripdata_2019-01.csv.gz

gzip -dk data/yellow_tripdata_2019-01.csv.gz

du -h data/yellow_tripdata_2019-01.csv
```

In [2]:
import logging
import time

import pandas as pd
import pyarrow as pa
from pyarrow import csv
import ray
from ray.data import DataContext

In [ ]:
# Restart Ray so this cell is reproducible after earlier notebook runs.
ray.shutdown()
ray.init(
    num_cpus=4,
    include_dashboard=False,
    logging_level=logging.ERROR,
    log_to_driver=False,
)

# Keep notebook output focused on results rather than execution progress.
data_context = DataContext.get_current()
data_context.enable_progress_bars = False
data_context.enable_operator_progress_bars = False
data_context.print_on_execution_start = False

print("Ray cluster resources:", ray.cluster_resources())
print("Currently available resources:", ray.available_resources())

2026-10-06 13:29:46,543	INFO worker.py:2015 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 


{'node:__internal_head__': 1.0, 'node:10.0.0.18': 1.0, 'memory': 9822789223.0, 'CPU': 4.0, 'object_store_memory': 4209766809.0, 'GPU': 1.0, 'accelerator_type:GeForce-RTX-3080-Ti-Laptop-GPU': 1.0}
{'node:__internal_head__': 1.0, 'node:10.0.0.18': 1.0, 'memory': 9822789223.0, 'CPU': 4.0, 'object_store_memory': 4209766809.0, 'GPU': 1.0, 'accelerator_type:GeForce-RTX-3080-Ti-Laptop-GPU': 1.0}


In [4]:
# The column begins with missing values, so declare its type explicitly to
# prevent PyArrow from incorrectly inferring it as a null-only column.
convert_options = csv.ConvertOptions(
    column_types={
        "congestion_surcharge": pa.float64(),
    }
)

# Include CSV reading and aggregation in the end-to-end measurement.
start = time.perf_counter()

dataset = ray.data.read_csv(
    "data/yellow_tripdata_2019-01.csv",
    convert_options=convert_options,
    override_num_blocks=32,
)

print(f"Dataset blocks: {dataset.num_blocks()}")

# Ray computes partial aggregates per block and combines them by group key.
# materialize() triggers the lazy execution plan and stores the result.
result = (
    dataset
    .groupby("passenger_count")
    .mean("total_amount")
    .materialize()
)

elapsed = time.perf_counter() - start

print(f"Ray end-to-end took: {elapsed:.2f} seconds")
result.show()

Ray resources: {'object_store_memory': 4380514713.0, 'node:10.0.0.18': 1.0, 'memory': 10221200999.0, 'CPU': 4.0, 'node:__internal_head__': 1.0, 'GPU': 1.0, 'accelerator_type:GeForce-RTX-3080-Ti-Laptop-GPU': 1.0}
Dataset blocks: <bound method Dataset.num_blocks of Dataset(num_rows=?, schema=Unknown schema)>


2026-10-06 13:30:47,605	WARNING __init__.py:28 -- Progress bars disabled. To enable, set `ray.data.DataContext.get_current().enable_progress_bars = True`.
2026-10-06 13:30:53,855	INFO hash_aggregate.py:161 -- Estimated memory requirement for aggregating aggregator (partitions=1, aggregators=1, dataset (estimate)=0.6GiB): shuffle=655.3MiB, output=655.3MiB, total=1310.5MiB, 
2026-10-06 13:31:16,642	INFO streaming_executor.py:365 -- ✔️  Dataset dataset_034bP3bsxouOeRQS3C4AZo_0 execution finished in 29.06 seconds
2026-10-06 13:31:16,716	INFO dataset.py:4088 -- Tip: Use `take_batch()` instead of `take() / show()` to return records in pandas or numpy batch format.
2026-10-06 13:31:16,771	INFO streaming_executor.py:365 -- ✔️  Dataset dataset_034bP4M0agHr3lMxYpIj9d_0 execution finished in 0.04 seconds


Ray Aggregation took 29.53 seconds
{'passenger_count': 0, 'mean(total_amount)': 18.6636575766095}
{'passenger_count': 1, 'mean(total_amount)': 15.609601335820825}
{'passenger_count': 2, 'mean(total_amount)': 15.83129398818425}
{'passenger_count': 3, 'mean(total_amount)': 15.60401463518482}
{'passenger_count': 4, 'mean(total_amount)': 15.650306636448251}
{'passenger_count': 5, 'mean(total_amount)': 15.546939865737002}
{'passenger_count': 6, 'mean(total_amount)': 15.437892147342522}
{'passenger_count': 7, 'mean(total_amount)': 48.27842105263158}
{'passenger_count': 8, 'mean(total_amount)': 64.1055172413793}
{'passenger_count': 9, 'mean(total_amount)': 31.09444444444445}


## Pandas baseline

For comparison, pandas performs the equivalent read and aggregation in a single process. This is an end-to-end comparison because both measurements include CSV parsing. On data that fits comfortably in memory, pandas may outperform Ray because it avoids distributed scheduling, object-store, and shuffle overhead.

In [5]:
# Start before read_csv so this measures the same end-to-end workload as Ray.
start = time.perf_counter()

df = pd.read_csv("data/yellow_tripdata_2019-01.csv")

# Select total_amount explicitly; mean()'s first positional argument is not
# a column name in pandas.
pandas_result = (
    df.groupby("passenger_count", as_index=False)["total_amount"]
    .mean()
)

elapsed = time.perf_counter() - start

print(f"Pandas end-to-end took: {elapsed:.2f} seconds")
pandas_result.head()

Pandas end-to-end took : 40.47 seconds


## Conclusion

Ray Data divides the CSV into blocks, schedules work across the configured CPUs, and combines partial results to produce the grouped mean. That execution model is valuable when data exceeds a single machine's memory, input is spread across many files, transformations are computationally expensive, or the workload must scale to a cluster.

For this example, the data fits in local memory and the aggregation has only a small number of passenger-count groups. Pandas can therefore be competitive or faster because its in-process implementation avoids Ray's task-scheduling, object-store, and shuffle overhead. The timings should be interpreted as an illustration of this workload on this machine—not as a general claim that one framework is always faster.

Possible next steps include reading several monthly taxi files, converting the CSV data to partitioned Parquet, applying more expensive transformations, and rerunning the experiment on a multi-node Ray cluster.